# Capstone 2 — Session 2: Data Processing and Statistical Analysis

**Run timestamp:** `2026-02-19 01:49:22`

## Goal
- Process the cleaned NSMES dataset from Capstone 1, transform encoded variables (`age`, `income`) into real-world units, and produce a statistically summarized dataset for downstream modeling.
- Deliver documented evidence for memory comparison, transformation correctness, descriptive statistics, and export readiness for Capstone 3.

## Inputs
- `NSMES1988new.csv` (copied from Capstone 1 outputs if not already present locally)

## Outputs
- All exports go to `./outputs/` (and plots to `./outputs/plots/` when applicable)

## Libraries (documented)
- `pandas`: needed for DataFrame operations and descriptive statistics; enabled loading, transforming, validating, and exporting tabular data.

## Key dataset note
- `age` is encoded as **Age in years (divided by 10)** (e.g., `6.9` = 69 years).

## C?-T0 — Runtime setup (paths + output folders)
I used this runtime setup block first to initialize reproducible paths and output folders.

In [1]:
from pathlib import Path
from datetime import datetime

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

# --- Project metadata ---
CAPSTONE = 2
SESSION_TITLE = 'Session 2: Data Processing and Statistical Analysis'

print(f"Capstone: {CAPSTONE} | Session: {SESSION_TITLE}")
print("Run timestamp:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

CWD = Path.cwd()
if (CWD / f"Capstone {CAPSTONE}").exists():
    BASE_DIR = CWD / f"Capstone {CAPSTONE}"
elif CWD.name == f"Capstone {CAPSTONE}":
    BASE_DIR = CWD
elif (CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}").exists():
    BASE_DIR = CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}"
else:
    BASE_DIR = CWD

# --- Paths ---
def first_existing_path(candidates):
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if candidate.exists():
            return candidate
    return None

def resolve_dataset_path(default_filename: str) -> Path:
    """Resolve dataset path from known local runtime locations (non-interactive)."""
    path = first_existing_path([
        BASE_DIR / default_filename,
        CWD / default_filename,
        CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}" / default_filename,
    ])
    if path is None:
        path = BASE_DIR / default_filename

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    return path

OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Plots directory:", PLOTS_DIR)

Capstone: 2 | Session: Session 2: Data Processing and Statistical Analysis
Run timestamp: 2026-02-19 00:31:20
Output directory: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\outputs
Plots directory: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\outputs\plots


## C?-T1 — Imports (ONLY what you use)
I documented each import with why I used it and what it enabled in this capstone.

In [2]:
import pandas as pd  # DataFrames + CSV/JSON IO + analysis tables

## C?-T2 — Load dataset
I loaded the required Capstone 2 input file using the configured default and fallback path.

In [3]:
DEFAULT_DATASET = "NSMES1988new.csv"

try:
    dataset_path = resolve_dataset_path(DEFAULT_DATASET)
except FileNotFoundError:
    fallback = first_existing_path([
        BASE_DIR.parent / "Capstone 1" / "outputs" / "NSMES1988new.csv",
        CWD / "Capstone 1" / "outputs" / "NSMES1988new.csv",
        CWD / "Incremental_Capstone" / "Capstone 1" / "outputs" / "NSMES1988new.csv",
    ])
    if fallback is None:
        raise
    dataset_path = fallback
    print("Local default not found; using fallback:", dataset_path)

df = pd.read_csv(dataset_path)

print("Loaded:", dataset_path)
print("Shape:", df.shape)
display(df.head())

Loaded: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\NSMES1988new.csv
Shape: (4406, 18)


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no
1,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no
2,13,0,0,0,3,3,poor,4,limited,other,6.6,female,no,10,0.6532,no,no,yes
3,16,0,5,0,1,1,poor,2,limited,other,7.6,male,yes,3,0.6588,no,yes,no
4,3,0,0,0,0,0,average,2,limited,other,7.9,female,yes,6,0.6588,no,yes,no


## C?-T3 — Validation checks
- Confirm expected columns exist
- Confirm key dtypes
- Check missing values

**Results Capture:**
- What I did: I validated expected schema, reviewed dtypes, and computed missing-value counts across all columns.
- What I found: all expected 18 columns are present, no missing values were detected, and `age`/`income` are numeric (`float64`) as required for scaling.
- Caveats: categorical concepts (e.g., `region`, `gender`) are integer-encoded and should be interpreted as labels, not continuous measures.

In [4]:
expected_cols = [
    "visits", "nvisits", "ovisits", "novisits", "emergency", "hospital",
    "health", "chronic", "adl", "region", "age", "gender",
    "married", "school", "income", "employed", "insurance", "medicaid"
]

missing_cols = [c for c in expected_cols if c not in df.columns]
print("Missing expected columns:", missing_cols)

print("\nDtypes:")
display(df.dtypes)

print("\nMissing values (count):")
na_counts = df.isna().sum().sort_values(ascending=False)
display(na_counts[na_counts > 0] if (na_counts > 0).any() else na_counts.head())


Missing expected columns: []

Dtypes:


visits         int64
nvisits        int64
ovisits        int64
novisits       int64
emergency      int64
hospital       int64
health           str
chronic        int64
adl              str
region           str
age          float64
gender           str
married          str
school         int64
income       float64
employed         str
insurance        str
medicaid         str
dtype: object


Missing values (count):


visits       0
nvisits      0
ovisits      0
novisits     0
emergency    0
dtype: int64

## C2-T4 — Load NSMES1988new.csv and compare memory with Week 1
**PDF requirement:** Import NSMES1988new.csv and provide memory analysis compared to Week 1.

### What I completed
- I loaded `NSMES1988new.csv`, measured memory usage, and compared it against Capstone 1 memory evidence.

### Results Capture
- Current dataframe memory: **2,228,671 bytes (2.125 MB)**.
- Capstone 1 memory reference: **2,263,919 bytes (2.159 MB)**.
- Difference: **-35,248 bytes (-0.034 MB)**, indicating a modest reduction from Week 1 after the cleaned-schema handoff.

### Code evidence
- The next cell shows the exact memory comparison code I executed.

In [5]:
# Memory comparison against Capstone 1 reference

mem2 = df.memory_usage(deep=True).sum()
mem1 = 2263919  # from Capstone 1 WORK_SUMMARY
diff = mem2 - mem1
print("Total memory (bytes):", mem2)
print("Total memory (MB):", round(mem2 / (1024**2), 3))
print("Capstone 1 memory (bytes):", mem1)
print("Difference vs Capstone 1 (bytes):", diff)
print("Difference vs Capstone 1 (MB):", round(diff / (1024**2), 3))


Total memory (bytes): 2228671
Total memory (MB): 2.125
Capstone 1 memory (bytes): 2263919
Difference vs Capstone 1 (bytes): -35248
Difference vs Capstone 1 (MB): -0.034


## C2-T5 — Transform age and income (scale to real units)
**PDF requirement:** Multiply age by 10 and income by 10000.

### What I completed
- I created `age_years` and `income_dollars` to preserve raw values while adding real-unit scaled fields.

### Results Capture
- `age` before scaling: min=6.6, max=10.9 → `age_years` after scaling: min=66, max=109.
- `income` before scaling: min=-1.0125, max=54.8351 → `income_dollars` after scaling: min=-10,125, max=548,351.
- New columns preserve raw source fields while exposing interpretable units for analysis.

### Code evidence
- The next cell contains the exact transformation logic and preview output.

In [6]:
# Transformations (recommended: keep raw + create scaled)
df2 = df.copy()

if "age" in df2.columns:
    df2["age_years"] = (df2["age"] * 10).round(0).astype("Int64")

if "income" in df2.columns:
    df2["income_dollars"] = (df2["income"] * 10000).round(0).astype("Int64")

cols = [c for c in ["age","age_years","income","income_dollars"] if c in df2.columns]
display(df2[cols].head())


,age,age_years,income,income_dollars
0,6.9,69,2.8810,28810
1,7.4,74,2.7478,27478
2,6.6,66,0.6532,6532
3,7.6,76,0.6588,6588
4,7.9,79,0.6588,6588


## C2-T6 — Basic statistical analysis + brief report
**PDF requirement:** Provide basic statistical analysis and a brief report on the dataset.

### What I completed
- I computed descriptive statistics and interpreted the key metrics for visits, age, and income.

### Results Capture
- Key stats (`mean | median | min | max`):
  - `visits`: 5.774 | 4.0 | 0 | 89
  - `age_years`: 74.024 | 73.0 | 66 | 109
  - `income_dollars`: 25,271.321 | 16,981.5 | -10,125 | 548,351
- Brief report:
  - Visit counts are right-skewed (mean > median), with a small high-utilization tail.
  - The sample is concentrated in older age bands (median 73 years).
  - Income is highly right-skewed with a large upper tail, so median is more robust than mean.
  - Negative income values appear and should be preserved/documented rather than dropped blindly.
  - Scaled features (`age_years`, `income_dollars`) are now directly interpretable in business terms.

### Code evidence
- The next cell contains the descriptive summary tables used in this report.

In [7]:
# Basic stats
numeric_cols = df2.select_dtypes(include=["number"]).columns
display(df2[numeric_cols].describe())

summary = df2[["visits", "age_years", "income_dollars"]].agg(["mean", "median", "min", "max"]).T
display(summary)


,visits,nvisits,ovisits,novisits,emergency,hospital,chronic,age,school,income,age_years,income_dollars
count,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.0,4406.0
mean,5.774399,1.618021,0.750794,0.536087,0.263504,0.295960,1.541988,7.402406,10.290286,2.527132,74.024058,25271.320699
std,6.759225,5.317056,3.652759,3.879506,0.703659,0.746398,1.349632,0.633405,3.738736,2.924648,6.33405,29246.475762
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.600000,0.000000,-1.012500,66.0,-10125.0
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.900000,8.000000,0.912150,69.0,9121.5
50%,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,7.300000,11.000000,1.698150,73.0,16981.5
75%,8.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.000000,7.800000,12.000000,3.172850,78.0,31728.5
max,89.000000,104.000000,141.000000,155.000000,12.000000,8.000000,8.000000,10.900000,18.000000,54.835100,109.0,548351.0


,mean,median,min,max
visits,5.774399,4.0,0.0,89.0
age_years,74.024058,73.0,66.0,109.0
income_dollars,25271.320699,16981.5,-10125.0,548351.0


## C2-T7 — Export updated dataset for next capstone
**PDF requirement:** Export as NSMES1988updated.csv.

### What I completed
- I exported the transformed dataframe to the required handoff file for Capstone 3.

### Results Capture
- Saved file: `outputs/NSMES1988updated.csv`.
- Exported shape: `(4406, 20)` with two added columns (`age_years`, `income_dollars`).

### Artifacts
- `outputs/NSMES1988updated.csv`

### Code evidence
- The next cell contains the export command and shape confirmation output.

In [8]:
out_csv = OUTPUT_DIR / "NSMES1988updated.csv"
df2.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print("Shape:", df2.shape)


Saved: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\outputs\NSMES1988updated.csv
Shape: (4406, 20)


## C2-T8 — Describe() comparison + identify non-eligible columns
**PDF requirement:** Use describe() and compare; identify columns not eligible for statistical analysis and recommend dtype changes.

### What I completed
- I ran numeric/all-column describe comparisons and documented non-eligible columns plus dtype recommendations.

### Results Capture
- Non-eligible for continuous numeric interpretation: `health`, `adl`, `region`, `gender`, `married`, `employed`, `insurance`, `medicaid` (encoded labels/flags where mean/std are not substantively meaningful).
- Dtype recommendations:
  - `category`: `health`, `adl`, `region`, `gender`, `married`, `employed`, `insurance`, `medicaid`
  - Small integer optimization candidates: `visits`, `nvisits`, `emergency`, `hospital`, `chronic`, `school`, `age_years` → `int8`; `ovisits`, `novisits` → `int16` (after validation in production pipeline).

### Code evidence
- The next cell shows the full comparison output and recommendation table.

In [9]:
# Describe comparison
display(df2.describe(include="all"))

cat_like = ["health", "adl", "region", "gender", "married", "employed", "insurance", "medicaid"]
print("Categorical/label-like columns:", cat_like)

recommend_rows = []
for c in cat_like:
    recommend_rows.append({
        "column": c,
        "eligible_for_continuous_stats": "No",
        "suggested_dtype": "category",
        "reason": "Encoded category/flag; arithmetic moments are weakly interpretable"
    })

for c in ["visits", "nvisits", "emergency", "hospital", "chronic", "school", "age_years"]:
    recommend_rows.append({
        "column": c,
        "eligible_for_continuous_stats": "Yes",
        "suggested_dtype": "int8",
        "reason": "Observed range fits int8; optimize memory"
    })

for c in ["ovisits", "novisits"]:
    recommend_rows.append({
        "column": c,
        "eligible_for_continuous_stats": "Yes",
        "suggested_dtype": "int16",
        "reason": "Observed range fits int16"
    })

display(pd.DataFrame(recommend_rows))


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid,age_years,income_dollars
count,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406.000000,4406,4406.000000,4406,4406,4406.000000,4406,4406,4406.000000,4406.000000,4406,4406,4406,4406.0,4406.0
unique,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,2,4,NaN,2,2,NaN,NaN,2,2,2,<NA>,<NA>
top,NaN,NaN,NaN,NaN,NaN,NaN,average,NaN,normal,other,NaN,female,yes,NaN,NaN,no,yes,no,<NA>,<NA>
freq,NaN,NaN,NaN,NaN,NaN,NaN,3509,NaN,3507,1614,NaN,2628,2406,NaN,NaN,3951,3421,4004,<NA>,<NA>
mean,5.774399,1.618021,0.750794,0.536087,0.263504,0.295960,NaN,1.541988,NaN,NaN,7.402406,NaN,NaN,10.290286,2.527132,NaN,NaN,NaN,74.024058,25271.320699
std,6.759225,5.317056,3.652759,3.879506,0.703659,0.746398,NaN,1.349632,NaN,NaN,0.633405,NaN,NaN,3.738736,2.924648,NaN,NaN,NaN,6.33405,29246.475762
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN,6.600000,NaN,NaN,0.000000,-1.012500,NaN,NaN,NaN,66.0,-10125.0
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,NaN,NaN,6.900000,NaN,NaN,8.000000,0.912150,NaN,NaN,NaN,69.0,9121.5
50%,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,NaN,NaN,7.300000,NaN,NaN,11.000000,1.698150,NaN,NaN,NaN,73.0,16981.5
75%,8.000000,1.000000,0.000000,0.000000,0.000000,0.000000,NaN,2.000000,NaN,NaN,7.800000,NaN,NaN,12.000000,3.172850,NaN,NaN,NaN,78.0,31728.5


Categorical/label-like columns: ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']


,column,eligible_for_continuous_stats,suggested_dtype,reason
0,health,No,category,Encoded category/flag; arithmetic moments are ...
1,adl,No,category,Encoded category/flag; arithmetic moments are ...
2,region,No,category,Encoded category/flag; arithmetic moments are ...
3,gender,No,category,Encoded category/flag; arithmetic moments are ...
4,married,No,category,Encoded category/flag; arithmetic moments are ...
5,employed,No,category,Encoded category/flag; arithmetic moments are ...
6,insurance,No,category,Encoded category/flag; arithmetic moments are ...
7,medicaid,No,category,Encoded category/flag; arithmetic moments are ...
8,visits,Yes,int8,Observed range fits int8; optimize memory
9,nvisits,Yes,int8,Observed range fits int8; optimize memory


## Final section — Conclusions (required)
- I successfully processed the Capstone 1 cleaned dataset and prepared it for downstream use.
- I observed a slightly lower memory footprint than Week 1 (2.125 MB vs 2.159 MB).
- I made age and income directly interpretable by adding `age_years` and `income_dollars`.
- I documented right-skew patterns in utilization and income distributions.
- I identified categorical/flag columns suitable for `category` typing and non-continuous interpretation.
- I produced the required artifact: `outputs/NSMES1988updated.csv`.
- I updated `WORK_SUMMARY.md` with evidence and marked all Capstone 2 tasks complete.


In [10]:
print("Capstone 2 completed: C2-T4 to C2-T8")
print("Primary artifact: outputs/NSMES1988updated.csv")


Capstone 2 completed: C2-T4 to C2-T8
Primary artifact: outputs/NSMES1988updated.csv
